# CROSCIM — GNN solver: how it works

Pure-geometry / schematic notebook (no cluster data or checkpoint needed —
matplotlib only) illustrating how the `MultiResGridGNN` backbone works and how
it plugs into the CROSCIM multi-resolution pipeline.

Source files illustrated here:
- `contrib/CROSCIM/solvers/GNN.py` — `MultiResGridGNN`, `SpatialGraphAttention`,
  `GNNBlock`, `GNNDownBlock`/`GNNUpBlock` (Graph-UNet pooling, `n_levels>1`)
- `contrib/CROSCIM/solvers/gnn_solver.py` — `GNNSolver`, `MultiResGNNSolvers`
- `contrib/CROSCIM/models/models_gnn.py` — `Lit4dVarNet_CROSCIM_GNN`
- `config/xp/CROSCIM/GNN_solvers/base_arctic_croscim_gnn_sit.yaml` — `gnn_config`

Sections:
1. Grid → graph: the 8-connected node graph
2. `SpatialGraphAttention` — one message-passing step
3. Full flat pipeline (`n_levels=1`, the original/checkpoint-compatible mode)
4. Graph-UNet pooling (`n_levels>1`) — lighter graphs, larger receptive field
5. Node count / receptive field: flat vs Graph-UNet, real patch size (256×256)
6. Multi-resolution integration — how `GNNSolver` plugs into x50/x10 CROSCIM


In [ ]:
import os, re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.lines import Line2D

FIGS_DIR = 'figs'
os.makedirs(FIGS_DIR, exist_ok=True)

def savefig_gnn(fig, name, dpi=600):
    """Save `fig` into FIGS_DIR as a publication-ready PNG, prefixed GNN_."""
    if not name.startswith('GNN'):
        name = f'GNN_{name}'
    if not name.lower().endswith('.png'):
        name += '.png'
    out_path = os.path.join(FIGS_DIR, name)
    fig.savefig(out_path, dpi=dpi, bbox_inches='tight')
    print(f"  saved {out_path}")

VALID_COLOR = '#2a78d6'
LAND_COLOR  = '#b7b19a'
EDGE_COLOR  = '#5a5a5a'
CENTER_COLOR = '#eb6834'
ATTN_COLOR  = '#1baf7a'


## 1. Grid → graph: the 8-connected node graph

Every pixel of the (H×W) patch is a *node*; a valid (ocean) pixel is connected
to its up to 8 immediate neighbours (3×3 window). Land / out-of-domain pixels
carry `valid_mask=0`: they still occupy a grid position but send/receive no
messages (masked to `-inf` before the attention softmax in
`SpatialGraphAttention`, see §2).

Crucially, `_gather_3x3()` in `GNN.py` does **not** build an explicit
`edge_index` (as a PyTorch-Geometric graph would) — it gathers the fixed 3×3
neighbourhood via `F.pad` + slicing, i.e. as a dense tensor operation on the
regular grid. The "graph" here is a fixed local-window structure, not a
generic sparse graph — this is what keeps the cost close to a plain
convolution (see §5).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRID -> GRAPH — 8-connected local neighbourhood, land pixels masked out
# ══════════════════════════════════════════════════════════════════════════════
N = 8  # small illustrative grid (real patches are 256x256, see SS5)
land = np.zeros((N, N), dtype=bool)
land[0:2, 5:8] = True   # a land blob in a corner, illustrative only
land[6:8, 0:2] = True

CENTER = (4, 4)  # (row, col) of the highlighted node

fig, ax = plt.subplots(figsize=(7, 7))

# nodes
for i in range(N):
    for j in range(N):
        is_land = land[i, j]
        is_center = (i, j) == CENTER
        color = LAND_COLOR if is_land else VALID_COLOR
        if is_center:
            color = CENTER_COLOR
        ax.scatter(j, -i, s=260 if is_center else 160, color=color,
                   edgecolor='white', linewidth=1.2, zorder=3)

# edges from the center node to its up-to-8 valid neighbours
ci, cj = CENTER
n_edges = 0
for di in (-1, 0, 1):
    for dj in (-1, 0, 1):
        if di == 0 and dj == 0:
            continue
        ni, nj = ci + di, cj + dj
        if not (0 <= ni < N and 0 <= nj < N):
            continue
        if land[ni, nj]:
            # masked edge: drawn dashed/faint, no message passed
            ax.plot([cj, nj], [-ci, -ni], color=LAND_COLOR, lw=1.2,
                    linestyle='--', alpha=0.6, zorder=1)
        else:
            ax.plot([cj, nj], [-ci, -ni], color=ATTN_COLOR, lw=2.0,
                    alpha=0.85, zorder=2)
            n_edges += 1

ax.set_xlim(-1, N)
ax.set_ylim(-N, 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title(
    f'Grid as an 8-connected graph — centre node has {n_edges} valid edges\n'
    '(land neighbours masked to $-\\infty$ before softmax, no message sent)',
    fontsize=12.5, fontweight='bold', pad=12,
)

legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=VALID_COLOR,
           markersize=11, label='Valid (ocean) node'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=LAND_COLOR,
           markersize=11, label='Invalid (land) node — masked out'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=CENTER_COLOR,
           markersize=13, label='Centre node (query)'),
    Line2D([0], [0], color=ATTN_COLOR, lw=2, label='Active edge (message passed)'),
    Line2D([0], [0], color=LAND_COLOR, lw=1.2, linestyle='--', label='Masked edge (no message)'),
]
ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.02),
          ncols=2, frameon=False, fontsize=10.5)

fig.tight_layout()
savefig_gnn(fig, 'grid_to_graph')
plt.show()


## 2. `SpatialGraphAttention` — one message-passing step

For every node *i*, `SpatialGraphAttention.forward()`:

1. Gathers the (up to 9, including self) neighbour features `neigh` and their
   validity `nmask` via `_gather_3x3`.
2. Concatenates `[center, neighbour]` and projects to a per-head score:
   `attn = attn_proj(concat(center, neigh))` → `(B,H,W,9,n_heads)`.
3. Masks invalid neighbours to `-inf`, softmaxes **over the 9 neighbours**
   (not over all nodes — this is the key difference from full self-attention).
4. Aggregates `val_proj(neigh)` weighted by the attention, per head.
5. Residual + `LayerNorm`.

This is a *windowed* attention (window = 3×3, fixed and local), architecturally
similar to a convolution with input-dependent (rather than fixed) weights —
its cost per node is O(9·C), **not** O(N·C) or O(N²) like full self-attention
over the whole patch.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SpatialGraphAttention — one node's neighbourhood attention (illustrative weights)
# ══════════════════════════════════════════════════════════════════════════════
offsets = [(-1, -1), (-1, 0), (-1, 1),
           ( 0, -1), ( 0, 0), ( 0, 1),
           ( 1, -1), ( 1, 0), ( 1, 1)]
valid_neighbor = np.array([True, True, True, True, True, True, False, True, True])  # (1,-1) masked = land

rng = np.random.default_rng(3)
raw_scores = rng.normal(size=9)
raw_scores[~valid_neighbor] = -np.inf
attn_w = np.exp(raw_scores - np.nanmax(raw_scores[valid_neighbor]))
attn_w[~valid_neighbor] = 0.0
attn_w = attn_w / attn_w.sum()  # softmax over the 9 (masked) neighbours

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={'width_ratios': [1, 1.1]})

# --- left: neighbourhood grid with attention-weighted arrows ---
ax = axes[0]
for (di, dj), w, is_valid in zip(offsets, attn_w, valid_neighbor):
    is_center = (di, dj) == (0, 0)
    color = CENTER_COLOR if is_center else (VALID_COLOR if is_valid else LAND_COLOR)
    ax.scatter(dj, -di, s=280 if is_center else 200, color=color,
               edgecolor='white', linewidth=1.2, zorder=3)
    if not is_center:
        if is_valid:
            ax.annotate('', xy=(0, 0), xytext=(dj, -di),
                        arrowprops=dict(arrowstyle='-|>', color=ATTN_COLOR,
                                        lw=1 + 7 * w, alpha=0.85,
                                        shrinkA=14, shrinkB=14))
            ax.text(dj * 1.32, -di * 1.32, f'{w:.2f}', ha='center', va='center',
                    fontsize=10, color=ATTN_COLOR, fontweight='bold')
        else:
            ax.text(dj * 1.32, -di * 1.32, 'masked', ha='center', va='center',
                    fontsize=9, color=LAND_COLOR, style='italic')
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Attention weights over the 3×3 neighbourhood\n(arrow width/label = softmax weight)',
             fontsize=11.5, fontweight='bold')

# --- right: attention weight bar chart, in raster order ---
ax = axes[1]
labels = [f'({di:+d},{dj:+d})' if (di, dj) != (0, 0) else '(self)' for di, dj in offsets]
colors = [LAND_COLOR if not v else (CENTER_COLOR if (di, dj) == (0, 0) else ATTN_COLOR)
          for (di, dj), v in zip(offsets, valid_neighbor)]
ax.bar(labels, attn_w, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('softmax attention weight')
ax.set_title('Per-neighbour attention weights\n(masked neighbour → weight 0, not shown to softmax)',
             fontsize=11.5, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)

fig.suptitle('SpatialGraphAttention — one message-passing step for the centre node',
             fontsize=13.5, fontweight='bold', y=1.02)
fig.tight_layout()
savefig_gnn(fig, 'spatial_graph_attention')
plt.show()


## 3. Full flat pipeline (`n_levels=1`)

`MultiResGridGNN.forward()` in flat mode (`n_levels=1`, the default and the
mode every currently-trained GNN checkpoint uses):

```
input (B, C_in, H, W), NaN on invalid nodes
   │  nan_to_num(0) + concat PositionalEncoding2D(lat, lon)
   ▼
input_proj  (1×1 Conv)  →  (B, hidden_dim, H, W)
   │
   ▼
GNNBlock × n_layers   (SpatialGraphAttention + FFN, residual+LayerNorm each)
   │
   ▼
output_proj (1×1 Conv)  →  (B, C_out, H, W)
   │  mask land → NaN
   ▼
output
```

Because every `GNNBlock` only reaches a node's 3×3 neighbourhood, the
receptive field grows **by exactly 1 hop per layer**: after `n_layers=6`
layers a node has only "seen" information from **13×13 pixels around it**
(6 hops each side + centre) — out of a 256×256 patch. This is the crux of the
"too big a graph, too small a reach" question: 65k nodes are processed every
layer, but each one only communicates locally.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FLAT PIPELINE (n_levels=1) — block diagram + receptive-field illustration
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={'width_ratios': [1.3, 1]})

# --- left: block-diagram flowchart ---
ax = axes[0]
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(10, 0)  # inverted: read top (input) -> bottom (output)

blocks = [
    ('input\n(B, C_in, H, W)\nNaN on land', 0.5, '#e8e6df'),
    ('+ PositionalEncoding2D\n(lat, lon)', 1.7, '#e8e6df'),
    ('input_proj\n1×1 Conv → hidden_dim', 2.9, VALID_COLOR),
    ('GNNBlock × n_layers\n(attention + FFN)', 4.4, ATTN_COLOR),
    ('output_proj\n1×1 Conv → C_out', 5.9, VALID_COLOR),
    ('mask land → NaN', 7.1, '#e8e6df'),
    ('output\n(B, C_out, H, W)', 8.3, '#e8e6df'),
]
box_w, box_h = 7.6, 0.85
for label, y, color in blocks:
    fc = color if color != ATTN_COLOR else '#d7f2e8'
    ec = ATTN_COLOR if color == ATTN_COLOR else '#8a8a8a'
    ax.add_patch(FancyBboxPatch((1.2, y - box_h / 2), box_w, box_h,
                                 boxstyle='round,pad=0.02,rounding_size=0.08',
                                 facecolor=fc, edgecolor=ec, linewidth=1.6, zorder=2))
    ax.text(1.2 + box_w / 2, y, label, ha='center', va='center',
            fontsize=10.5, fontweight='bold', zorder=3)
for k in range(len(blocks) - 1):
    y0 = blocks[k][1] + box_h / 2
    y1 = blocks[k + 1][1] - box_h / 2
    ax.annotate('', xy=(1.2 + box_w / 2, y1), xytext=(1.2 + box_w / 2, y0),
                arrowprops=dict(arrowstyle='-|>', color='#5a5a5a', lw=1.6))
ax.set_title('Flat pipeline (n_levels=1)', fontsize=13, fontweight='bold', pad=10)

# --- right: receptive field after each layer (concentric rings) ---
ax = axes[1]
n_layers = 6
patch_extent = 256
for hop in range(n_layers, 0, -1):
    size = 2 * hop + 1
    frac = size / patch_extent
    alpha = 0.15 + 0.10 * (n_layers - hop)
    ax.add_patch(mpatches.Rectangle((0.5 - frac / 2, 0.5 - frac / 2), frac, frac,
                                     facecolor=ATTN_COLOR, edgecolor='none', alpha=alpha,
                                     transform=ax.transAxes, zorder=2))
ax.add_patch(mpatches.Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='#8a8a8a',
                                 linewidth=1.5, transform=ax.transAxes, zorder=1))
ax.scatter([0.5], [0.5], s=60, color=CENTER_COLOR, zorder=4, transform=ax.transAxes)
ax.text(0.5, 0.02, f'full patch = {patch_extent}×{patch_extent} px', ha='center', va='bottom',
        fontsize=9.5, transform=ax.transAxes, color='#5a5a5a')
ax.text(0.5, 0.5 + (2 * n_layers + 1) / patch_extent / 2 + 0.03,
        f'receptive field after\n{n_layers} layers = {2*n_layers+1}×{2*n_layers+1} px',
        ha='center', va='bottom', fontsize=10, fontweight='bold', color=ATTN_COLOR,
        transform=ax.transAxes)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Receptive field vs full patch\n(grows by only 1 hop / layer)',
             fontsize=12, fontweight='bold', pad=10)

fig.tight_layout()
savefig_gnn(fig, 'flat_pipeline')
plt.show()


## 4. Graph-UNet pooling (`n_levels>1`) — lighter graphs, larger receptive field

New `n_levels` parameter of `MultiResGridGNN` (default `1` = the flat mode
above, exactly reproduced — existing checkpoints keep working unchanged).
When `n_levels>1`, `GNN.py` turns the flat stack into a **Graph-UNet**:

```
encoder (× n_levels):  GNNBlock(s) at current resolution → strided-conv ×2 downsample
                        (features zeroed on invalid nodes before pooling;
                         mask itself max-pooled: coarse cell valid if any
                         finer cell was valid)
bottleneck:             GNNBlock(s) at the coarsest resolution
decoder (× n_levels):  upsample ×2 (nearest) → concat with matching skip
                        connection → 1×1 fuse conv → GNNBlock(s)
```

Two effects, both addressing the "graphes énormes" concern directly:
- **Fewer active nodes deeper in the network** (each pooling step divides the
  node count by 4), so most of the depth (bottleneck + surrounding levels) is
  spent on a much smaller graph.
- **Receptive field grows geometrically, not linearly**: a hop at the coarsest
  level covers `2^n_levels` original pixels, so the *same* number of
  attention layers now reaches a much larger physical area — exactly how a
  CNN U-Net gets cheap global context via downsampling.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRAPH-UNET (n_levels>1) — pyramid diagram with per-level node counts
# ══════════════════════════════════════════════════════════════════════════════
H0 = W0 = 256          # actual GNN_solvers patch size (patch_dims: yc=256, xc=256)
N_LEVELS = 3            # illustrative choice

sizes = [(H0 // (2 ** l), W0 // (2 ** l)) for l in range(N_LEVELS + 1)]
node_counts = [h * w for h, w in sizes]

fig, ax = plt.subplots(figsize=(9, 7))
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

n_stages = N_LEVELS + 1  # encoder levels + bottleneck
y_positions = np.linspace(8.6, 1.0, n_stages)
box_w_max = 6.5

for l, (y, (h, w), n) in enumerate(zip(y_positions, sizes, node_counts)):
    frac = np.sqrt(n / node_counts[0])
    bw = max(box_w_max * frac, 1.1)
    label_stage = 'bottleneck' if l == N_LEVELS else f'encoder level {l}'
    color = CENTER_COLOR if l == N_LEVELS else VALID_COLOR
    ax.add_patch(FancyBboxPatch((5 - bw / 2, y - 0.45), bw, 0.9,
                                 boxstyle='round,pad=0.02,rounding_size=0.08',
                                 facecolor=color, alpha=0.75, edgecolor='#3a3a3a',
                                 linewidth=1.4, zorder=2))
    ax.text(5, y, f'{h}×{w}  =  {n:,} nodes', ha='center', va='center',
            fontsize=10.5, fontweight='bold', color='white', zorder=3)
    ax.text(0.3, y, label_stage, ha='left', va='center', fontsize=10, color='#3a3a3a')
    if l < N_LEVELS:
        y_next = y_positions[l + 1]
        ax.annotate('', xy=(5 - bw / 4, y_next + 0.55), xytext=(5 - bw / 4, y - 0.5),
                    arrowprops=dict(arrowstyle='-|>', color=ATTN_COLOR, lw=1.8))
        ax.text(5 - bw / 4 - 0.35, (y + y_next) / 2, 'pool ×2', ha='right', va='center',
                fontsize=9, color=ATTN_COLOR, rotation=90)
        ax.annotate('', xy=(5 + bw / 4 + 0.2, y - 0.5), xytext=(5 + bw / 4 + 0.2, y_next + 0.55),
                    arrowprops=dict(arrowstyle='-|>', color='#8a5fd6', lw=1.8))
        ax.text(5 + bw / 4 + 0.5, (y + y_next) / 2, 'unpool ×2\n+ skip', ha='left', va='center',
                fontsize=9, color='#8a5fd6', rotation=-90)

ax.set_title(
    f'Graph-UNet, n_levels={N_LEVELS} — 256×256 patch\n'
    f'{node_counts[0]:,} nodes at input → {node_counts[-1]:,} nodes at the bottleneck '
    f'({node_counts[0] // node_counts[-1]}× fewer)',
    fontsize=12.5, fontweight='bold', pad=14,
)
fig.tight_layout()
savefig_gnn(fig, 'graph_unet_pyramid')
plt.show()


## 5. Node count & receptive field: flat vs Graph-UNet (real 256×256 patch)

Quantitative version of the qualitative diagrams above, using the actual GNN
patch size (`patch_dims: {yc: 256, xc: 256}` in
`GNN_solvers/base_arctic_croscim_gnn_sit.yaml`) and `n_layers=6` (per-level
block count is kept the same in the Graph-UNet, `gnn_config.n_layers`).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# QUANTITATIVE COMPARISON — node count per level & receptive field, flat vs Graph-UNet
# ══════════════════════════════════════════════════════════════════════════════
H0 = W0 = 256
N_LAYERS = 6
LEVELS_RANGE = [0, 1, 2, 3]  # 0 = flat (no pooling); n_levels=3 keeps the bottleneck at 32x32, still well-posed

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- left: total processed-node budget (sum over all levels incl. skips) ---
ax = axes[0]
budgets = []
for n_levels in LEVELS_RANGE:
    if n_levels == 0:
        budget = H0 * W0 * N_LAYERS  # flat: n_layers blocks all at full res
    else:
        # encoder (n_layers blocks/level) + bottleneck (n_layers) + decoder (n_layers/level)
        enc = sum((H0 // 2 ** l) * (W0 // 2 ** l) * N_LAYERS for l in range(n_levels))
        bottleneck = (H0 // 2 ** n_levels) * (W0 // 2 ** n_levels) * N_LAYERS
        dec = sum((H0 // 2 ** l) * (W0 // 2 ** l) * N_LAYERS for l in range(n_levels))
        budget = enc + bottleneck + dec
    budgets.append(budget)
bars = ax.bar([str(l) if l else 'flat' for l in LEVELS_RANGE],
              np.array(budgets) / 1e6, color=[CENTER_COLOR] + [ATTN_COLOR] * (len(LEVELS_RANGE) - 1),
              edgecolor='black', linewidth=0.9)
for b, v in zip(bars, budgets):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f'{v/1e6:.1f}M',
            ha='center', va='bottom', fontsize=9.5)
ax.set_xlabel('n_levels')
ax.set_ylabel('total (node × GNNBlock) evaluations, millions')
ax.set_title('Compute proxy: total node-block evaluations\nper forward pass (256×256 patch, 6 blocks/level)',
             fontsize=11.5, fontweight='bold')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)

# --- right: receptive field (pixels) ---
ax = axes[1]
rf_flat = [2 * N_LAYERS * 1 + 1] * len(LEVELS_RANGE)  # constant: flat RF, independent of n_levels axis (reference line)
rf_unet = []
for n_levels in LEVELS_RANGE:
    if n_levels == 0:
        rf_unet.append(2 * N_LAYERS + 1)
        continue
    # each encoder level of N_LAYERS blocks adds N_LAYERS hops *at that level's resolution*
    # (i.e. N_LAYERS * 2**level pixels); bottleneck adds N_LAYERS hops at the coarsest resolution;
    # decoder levels add negligible extra reach (already captured via skips) -- lower-bound estimate.
    rf = 1
    for l in range(n_levels):
        rf += 2 * N_LAYERS * (2 ** l)
    rf += 2 * N_LAYERS * (2 ** n_levels)  # bottleneck hops, coarsest resolution
    rf_unet.append(rf)
ax.plot(LEVELS_RANGE, rf_flat, 'o--', color='#8a8a8a', label='flat (n_levels=0), reference')
ax.plot(LEVELS_RANGE, rf_unet, 'o-', color=ATTN_COLOR, linewidth=2.2, label='Graph-UNet')
for x, y in zip(LEVELS_RANGE, rf_unet):
    ax.annotate(f'{y}px', (x, y), textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=9)
ax.axhline(H0, color=CENTER_COLOR, linestyle=':', linewidth=1.4)
ax.text(0.05, H0, f'full patch = {H0}px', color=CENTER_COLOR, fontsize=9,
        va='bottom', transform=ax.get_yaxis_transform())
ax.set_xlabel('n_levels')
ax.set_ylabel('receptive field (pixels)')
ax.set_yscale('log')
ax.set_title('Receptive field grows geometrically with n_levels\n(flat: fixed at 13px regardless of patch size)',
             fontsize=11.5, fontweight='bold')
ax.legend(frameon=False, fontsize=10)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)

fig.tight_layout()
savefig_gnn(fig, 'flat_vs_graphunet_quantitative')
plt.show()


## 6. Multi-resolution integration — how the GNN plugs into CROSCIM

`MultiResGNNSolvers` (`gnn_solver.py`) is a thin `nn.ModuleDict` container of
one independent `GNNSolver` per resolution (`solver_x50`, `solver_x10`), each
wrapping its own `MultiResGridGNN` instance — same interface as
`ConsistencyGradSolvers` / the other multi-res solver containers, so
`Lit4dVarNet_CROSCIM_GNN` (`models_gnn.py`) reuses the **exact same** shared
multi-resolution coupling machinery in `models.py` as every other CROSCIM
solver (UNet, 4DVarNet, CM, FM): the coarse (x50) prediction is interpolated
onto the fine (x10) grid and used as an anomaly baseline that the fine solver
only has to correct — the GNN itself is a drop-in backbone, unaware of the
multi-res logic around it.

This is orthogonal to §4/§5: `n_levels` only changes what happens *inside*
one resolution's `MultiResGridGNN`, independently at x50 and x10 (each has
its own 256×256 patch and its own pooling depth if enabled).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MULTI-RESOLUTION INTEGRATION — x50 -> interpolate -> anomaly baseline -> x10
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.axis('off')
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)

def box(x, y, w, h, label, color, fontsize=10.5, textcolor='black'):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.08',
                                 facecolor=color, edgecolor='#3a3a3a', linewidth=1.5, zorder=2))
    ax.text(x + w / 2, y + h / 2, label, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color=textcolor, zorder=3)

def arrow(x0, y0, x1, y1, color='#5a5a5a', label=None, rad=0.0):
    ax.add_artist(FancyArrowPatch((x0, y0), (x1, y1), color=color, lw=1.8,
                                   arrowstyle='-|>', mutation_scale=15,
                                   connectionstyle=f'arc3,rad={rad}', zorder=1))
    if label:
        ax.text((x0 + x1) / 2, (y0 + y1) / 2 + 0.25, label, ha='center', fontsize=9.2, color=color)

# x50 branch
box(0.3, 3.6, 2.4, 1.1, 'x50 patch\n256×256 grid', '#e8e6df')
box(0.3, 1.9, 2.4, 1.1, 'GNNSolver (x50)\nMultiResGridGNN', VALID_COLOR, textcolor='white')
arrow(1.5, 3.6, 1.5, 3.0)
box(0.3, 0.2, 2.4, 1.1, 'pred_x50', '#d7f2e8')
arrow(1.5, 1.9, 1.5, 1.3)

# interpolation / anomaly coupling (shared models.py machinery)
box(3.5, 2.6, 3.2, 1.6,
    'interpolate_torch(pred_x50 → x10 grid)\nused as anomaly baseline\n'
    '(update_batch_as_anomaly, models.py)\n— identical for UNet / CM / FM / GNN',
    '#f3e6c9', fontsize=9.2)
arrow(2.7, 0.75, 3.5, 3.0, label='pred_x50', rad=0.25)

# x10 branch
box(7.6, 3.6, 2.4, 1.1, 'x10 patch\n256×256 grid\n+ interpolated x50 baseline', '#e8e6df', fontsize=9.5)
box(7.6, 1.9, 2.4, 1.1, 'GNNSolver (x10)\nMultiResGridGNN', VALID_COLOR, textcolor='white')
arrow(6.7, 3.4, 7.6, 3.9, label='anomaly\nbaseline', rad=-0.15)
arrow(8.8, 3.6, 8.8, 3.0)
box(7.6, 0.2, 2.4, 1.1, 'pred_x10\n(fine correction\n+ x50 baseline)', '#d7f2e8', fontsize=9.5)
arrow(8.8, 1.9, 8.8, 1.3)

ax.set_title(
    'Multi-resolution coupling (MultiResGNNSolvers) — coarse-to-fine anomaly correction\n'
    'GNN backbone is a drop-in replacement; the coupling logic is shared with every other CROSCIM solver',
    fontsize=12, fontweight='bold', pad=14,
)
fig.tight_layout()
savefig_gnn(fig, 'multires_integration')
plt.show()
